In [1]:
import requests

In [2]:
def get_vehicle_details(vin):
    pass

In [3]:

# API call code

import requests

def get_vehicle_details(vin):
    url = f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValues/{vin}?format=json"
    response = requests.get(url)
    data = response.json()
    return data["Results"][0]

In [4]:
vehicle = get_vehicle_details("1HGCM82633A004352")
vehicle

{'ABS': '',
 'ActiveSafetySysNote': '',
 'AdaptiveCruiseControl': '',
 'AdaptiveDrivingBeam': '',
 'AdaptiveHeadlights': '',
 'AdditionalErrorText': '',
 'AirBagLocCurtain': '1st and 2nd Rows',
 'AirBagLocFront': '1st Row (Driver and Passenger)',
 'AirBagLocKnee': '',
 'AirBagLocSeatCushion': '',
 'AirBagLocSide': '1st Row (Driver and Passenger)',
 'AutoReverseSystem': '',
 'AutomaticPedestrianAlertingSound': '',
 'AxleConfiguration': '',
 'Axles': '',
 'BasePrice': '',
 'BatteryA': '',
 'BatteryA_to': '',
 'BatteryCells': '',
 'BatteryInfo': '',
 'BatteryKWh': '',
 'BatteryKWh_to': '',
 'BatteryModules': '',
 'BatteryPacks': '',
 'BatteryType': '',
 'BatteryV': '',
 'BatteryV_to': '',
 'BedLengthIN': '',
 'BedType': 'Not Applicable',
 'BlindSpotIntervention': '',
 'BlindSpotMon': '',
 'BodyCabType': 'Not Applicable',
 'BodyClass': 'Coupe',
 'BrakeSystemDesc': '',
 'BrakeSystemType': '',
 'BusFloorConfigType': 'Not Applicable',
 'BusLength': '',
 'BusType': 'Not Applicable',
 'CAN_AACN

In [5]:
important_vehicle_info = {
    "make": vehicle.get("Make"),
    "model": vehicle.get("Model"),
    "year": vehicle.get("ModelYear"),
    "body_type": vehicle.get("BodyClass"),
}

important_vehicle_info

{'make': 'HONDA', 'model': 'Accord', 'year': '2003', 'body_type': 'Coupe'}

In [6]:
vehicle_year = int(important_vehicle_info["year"])

if vehicle_year < 2015:
    risk_level = "HIGH"
else:
    risk_level = "NORMAL"
risk_level

'HIGH'

In [7]:
clean_vehicle_info = {
    "make": vehicle.get("Make"),
    "model": vehicle.get("Model"),
    "year": vehicle.get("ModelYear"),
    "body_type": vehicle.get("BodyClass"),
}

clean_vehicle_info

{'make': 'HONDA', 'model': 'Accord', 'year': '2003', 'body_type': 'Coupe'}

In [8]:
import pandas as pd
contracts_df = pd.read_csv("../data/sample_car_contracts.csv")
contracts_df.head()

,id,customer_name,contract_type,vehicle_type,monthly_emi,interest_rate,tenure_months,clause_summary,risk_flag,issue_type,recommended_action
0,1,Rahul Mehta,Car Loan,Sedan,18500,9.5,48,Prepayment allowed only after 24 months with 5...,medium,High prepayment charges,Highlight prepayment penalty to user and sugge...
1,2,Anita Rao,Car Lease,SUV,22000,0.0,36,Lessee must pay for all maintenance and insurance,low,Standard maintenance clause,"No action, just explain maintenance responsibi..."
2,3,James Wilson,Car Loan,Hatchback,14500,11.2,60,Late payment fee of 3% per month on outstandin...,high,Aggressive late fee,Flag clause and suggest user request cap on la...
3,4,Meena Iyer,Car Lease,Sedan,21000,0.0,24,"Excess mileage charge of ₹12 per km over 15,00...",medium,High excess mileage rate,Warn user about extra mileage charges and reco...
4,5,Arjun Patel,Car Loan,SUV,27500,10.8,72,Floating interest rate linked to lender's inte...,high,Unclear interest benchmark,Explain floating rate risk and suggest asking ...


In [9]:

# Data Enrichment

def enrich_contract_with_vehicle(contract_row):
    contract_id = contract_row.get("contract_id")
    customer_name = contract_row.get("customer_name")
    vin = contract_row.get("vin")

    # If VIN is missing, do NOT call API
    if pd.isna(vin):
        return {
            "contract_id": contract_id,
            "customer_name": customer_name,
            "vin": None,
            "vehicle": None
        }

    vehicle = get_vehicle_details(vin)

    return {
        "contract_id": contract_id,
        "customer_name": customer_name,
        "vin": vin,
        "vehicle": {
            "make": vehicle.get("Make"),
            "model": vehicle.get("Model"),
            "year": vehicle.get("ModelYear")
        }
    }

In [10]:
combined_records = []

for _, row in contracts_df.iterrows():
    combined_records.append(enrich_contract_with_vehicle(row))

combined_records[:1]

[{'contract_id': None,
  'customer_name': 'Rahul Mehta',
  'vin': None,
  'vehicle': None}]

In [11]:

contracts_df.columns

Index(['id', 'customer_name', 'contract_type', 'vehicle_type', 'monthly_emi',
       'interest_rate', 'tenure_months', 'clause_summary', 'risk_flag',
       'issue_type', 'recommended_action'],
      dtype='object')

In [12]:
import pandas as pd 

contracts_df = pd.read_csv("../data/sample_car_contracts.csv")
contracts_df.head()

,id,customer_name,contract_type,vehicle_type,monthly_emi,interest_rate,tenure_months,clause_summary,risk_flag,issue_type,recommended_action
0,1,Rahul Mehta,Car Loan,Sedan,18500,9.5,48,Prepayment allowed only after 24 months with 5...,medium,High prepayment charges,Highlight prepayment penalty to user and sugge...
1,2,Anita Rao,Car Lease,SUV,22000,0.0,36,Lessee must pay for all maintenance and insurance,low,Standard maintenance clause,"No action, just explain maintenance responsibi..."
2,3,James Wilson,Car Loan,Hatchback,14500,11.2,60,Late payment fee of 3% per month on outstandin...,high,Aggressive late fee,Flag clause and suggest user request cap on la...
3,4,Meena Iyer,Car Lease,Sedan,21000,0.0,24,"Excess mileage charge of ₹12 per km over 15,00...",medium,High excess mileage rate,Warn user about extra mileage charges and reco...
4,5,Arjun Patel,Car Loan,SUV,27500,10.8,72,Floating interest rate linked to lender's inte...,high,Unclear interest benchmark,Explain floating rate risk and suggest asking ...


In [13]:

contracts_df["vin"] = "1HGCM82633A004352"

In [14]:
contracts_df[["vin"]].head()

,vin
0,1HGCM82633A004352
1,1HGCM82633A004352
2,1HGCM82633A004352
3,1HGCM82633A004352
4,1HGCM82633A004352


In [15]:

contracts_df.to_csv("../Assignment/sample_car_contracts_with_vin.csv", index=False)